# Registered frequency-sensitive head ablation — Pythia-2.8B

This notebook runs the held-out causal test in preregistration 0003 and its
2026-09-04 controls amendment. It uses 25 compounds to select five heads and
tests them on the other 24 compounds. The test compounds never influence
head selection.

The outcome is change in the next-token distribution, measured by KL
divergence. The notebook does not assume in advance that the result is null.


## Setup

In [ ]:
# Cell 0: Environment detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 1: Colab only — install pinned dependencies
# Restart the runtime after running this cell, then continue with Cell 2.
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6
    %pip install -q scipy

In [ ]:
# Cell 2: Project root and path setup
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")
    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"
    PROJECT_ROOT = Path("/content") / repo_name
    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Cell 3: Imports and device
import json
import math
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from scipy.stats import spearmanr
from transformer_lens import HookedTransformer
from src.ablation_manifest import write_registered_ablation_manifest
from src.qk_ov import ablate_head_sets_at_position

device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Cell 4: Frozen settings
model_name = "pythia-2.8b"
CONDITION = "natural"
RANDOM_SEED = 20260813
N_RANDOM_SETS = 100
N_PERMUTATIONS = 10_000
assert model_name == "pythia-2.8b" and CONDITION == "natural"
print("Frozen settings loaded.")

In [ ]:
# Cell 5: Load Pythia-2.8B
model = HookedTransformer.from_pretrained(f"EleutherAI/{model_name}")
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads per layer: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

## Freeze the compounds and head sets

This cell reads the split and candidate heads produced by the completed effective-binding analysis. It also saves all 100 random sets before any ablation result is summarized.


In [ ]:
# Cell 6: Load the saved split and freeze every head set
analysis_dir = PROJECT_ROOT / "results" / "analysis"
split_path = analysis_dir / "effective_binding_compound_split.csv"
candidate_path = analysis_dir / f"effective_binding_head_candidates_{CONDITION}.csv"
amendment_path = PROJECT_ROOT / "docs" / "preregistrations" / "0003-amendment-2026-09-04-ablation-controls.md"
for required_path in (split_path, candidate_path, amendment_path):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

split = pd.read_csv(split_path).sort_values("shuffle_position")
if set(split["random_seed"]) != {RANDOM_SEED}:
    raise ValueError("Saved compound split does not use the registered seed.")
if split["compound"].nunique() != 49:
    raise ValueError("Expected 49 unique compounds in the saved split.")
if (split["split"] == "selection").sum() != 25 or (split["split"] == "test").sum() != 24:
    raise ValueError("Expected a 25-selection / 24-test split.")
test_names = split.loc[split["split"] == "test", "compound"].tolist()

candidates = pd.read_csv(candidate_path)
chosen = candidates[(candidates["family"] == "pythia") & (candidates["model"] == model_name)].sort_values("candidate_rank")
if chosen["candidate_rank"].tolist() != [1, 2, 3, 4, 5]:
    raise ValueError("Expected exactly five ranked Pythia-2.8B candidate heads.")
selected_heads = [(int(layer), int(head)) for layer, head in zip(chosen["layer"], chosen["head"])]

late_start = math.ceil(2 * model.cfg.n_layers / 3)
selected_set = set(selected_heads)
eligible_random_heads = [(layer, head) for layer in range(late_start, model.cfg.n_layers) for head in range(model.cfg.n_heads) if (layer, head) not in selected_set]
rng_controls = np.random.default_rng(RANDOM_SEED)
random_head_sets, seen_sets = [], set()
while len(random_head_sets) < N_RANDOM_SETS:
    indices = rng_controls.choice(len(eligible_random_heads), size=5, replace=False)
    sampled = tuple(sorted(eligible_random_heads[int(i)] for i in indices))
    if sampled not in seen_sets:
        seen_sets.add(sampled)
        random_head_sets.append(list(sampled))
positive_control_heads = [(model.cfg.n_layers - 2, head) for head in range(model.cfg.n_heads)]

output_dir = PROJECT_ROOT / "results" / "ablation" / "frequency_heads" / model_name
output_dir.mkdir(parents=True, exist_ok=True)
head_sets_path = output_dir / f"{model_name}-{CONDITION}-registered-head-sets.csv"
head_set_rows = [
    {"set_id": "selected", "role": "selected", "heads": json.dumps(selected_heads), "n_heads": 5},
    {"set_id": "empty", "role": "empty_control", "heads": "[]", "n_heads": 0},
    {"set_id": "positive_control", "role": "positive_control", "heads": json.dumps(positive_control_heads), "n_heads": len(positive_control_heads)},
]
head_set_rows.extend({"set_id": f"random_{i:03d}", "role": "random_control", "heads": json.dumps(heads), "n_heads": 5} for i, heads in enumerate(random_head_sets, start=1))
pd.DataFrame(head_set_rows).to_csv(head_sets_path, index=False)
print("Selected heads:", selected_heads)
print("Held-out compounds:", len(test_names))
print("Late-layer random pool:", len(eligible_random_heads), "heads")
print("Saved 100 random sets before ablation:", head_sets_path)

## Run the selected, empty, and positive-control conditions

The empty and positive controls are gates. If either fails, this notebook stops before the expensive random-control run.


In [ ]:
# Cell 7: Run the primary set and the two gate controls
with open(PROJECT_ROOT / "data" / "binding" / "accessibility.yaml", "r") as handle:
    all_cases = yaml.safe_load(handle)["compounds"]
case_by_name = {case["name"]: case for case in all_cases}
missing_cases = sorted(set(test_names) - set(case_by_name))
if missing_cases:
    raise ValueError(f"Held-out compounds missing from accessibility.yaml: {missing_cases}")
test_cases = [case_by_name[name] for name in test_names]
frequency = pd.read_csv(PROJECT_ROOT / "results" / "frequency" / "frequency_table.csv")[["compound", "bigram_count"]].drop_duplicates()
primary_rows = []
primary_sets = {"selected": selected_heads, "empty": [], "positive_control": positive_control_heads}
for i, case in enumerate(test_cases, start=1):
    print(f"{i:2d}/{len(test_cases)}  {case['name']}")
    condition_results = ablate_head_sets_at_position(model, case["prompt"], primary_sets, case["word2"])
    row = {"compound": case["name"], "model": model_name, "condition": CONDITION, "selected_heads": json.dumps(selected_heads), "positive_control_heads": json.dumps(positive_control_heads)}
    for label, result in condition_results.items():
        row[f"{label}_kl"] = result["kl_base_to_ablated"]
        row[f"{label}_top_changed"] = result["baseline_top"][0][0] != result["ablated_top"][0][0]
    row["baseline_top"] = json.dumps(condition_results["selected"]["baseline_top"])
    row["selected_ablated_top"] = json.dumps(condition_results["selected"]["ablated_top"])
    row["positive_control_ablated_top"] = json.dumps(condition_results["positive_control"]["ablated_top"])
    primary_rows.append(row)
primary_results = pd.DataFrame(primary_rows).merge(frequency, on="compound", how="left", validate="one_to_one")
primary_results["log_frequency"] = np.log1p(primary_results["bigram_count"])
primary_output_path = output_dir / f"{model_name}-{CONDITION}-registered-primary.csv"
primary_results.to_csv(primary_output_path, index=False)
empty_max = float(primary_results["empty_kl"].abs().max())
positive_max = float(primary_results["positive_control_kl"].max())
if not np.isfinite(primary_results[["empty_kl", "positive_control_kl"]].to_numpy()).all():
    raise RuntimeError("A gate control produced a non-finite KL value. Stop and audit.")
if empty_max > 1e-8:
    raise RuntimeError(f"Empty-set gate failed: maximum KL was {empty_max:.3g}.")
if positive_max <= 1e-6:
    raise RuntimeError(f"Positive-control gate failed: maximum KL was {positive_max:.3g}.")
print(f"Saved: {primary_output_path}")
print(f"Empty-set gate passed (maximum KL {empty_max:.3g}).")
print(f"Positive-control gate passed (maximum KL {positive_max:.3g}).")
display(primary_results.head())

## Run 100 random five-head controls

This is the long cell. It saves a checkpoint after every compound. If the session stops, rerun the setup cells and then this cell; compounds with all 100 saved controls are skipped.


In [ ]:
# Cell 8: Run the random controls with per-compound checkpoints
random_output_path = output_dir / f"{model_name}-{CONDITION}-registered-random-controls.csv"
random_set_map = {f"random_{i:03d}": heads for i, heads in enumerate(random_head_sets, start=1)}
if random_output_path.exists():
    existing_random = pd.read_csv(random_output_path)
    required_columns = {"compound", "set_id", "heads", "kl", "top_changed"}
    if not required_columns.issubset(existing_random.columns):
        raise ValueError("Existing random-control checkpoint has the wrong schema.")
    expected_heads = {key: json.dumps(value) for key, value in random_set_map.items()}
    for row in existing_random[["set_id", "heads"]].drop_duplicates().itertuples(index=False):
        if row.set_id not in expected_heads or row.heads != expected_heads[row.set_id]:
            raise ValueError("Existing checkpoint does not match the frozen random head sets.")
    counts = existing_random.groupby("compound")["set_id"].nunique()
    completed_compounds = set(counts[counts == N_RANDOM_SETS].index)
    random_rows = existing_random[existing_random["compound"].isin(completed_compounds)].to_dict("records")
    print(f"Resuming with {len(completed_compounds)} compounds already complete.")
else:
    completed_compounds, random_rows = set(), []
frequency_by_compound = frequency.set_index("compound")["bigram_count"].to_dict()
for i, case in enumerate(test_cases, start=1):
    if case["name"] in completed_compounds:
        print(f"{i:2d}/{len(test_cases)}  {case['name']} — already saved")
        continue
    print(f"{i:2d}/{len(test_cases)}  {case['name']} — running 100 sets")
    condition_results = ablate_head_sets_at_position(model, case["prompt"], random_set_map, case["word2"])
    for set_id, result in condition_results.items():
        count = frequency_by_compound[case["name"]]
        random_rows.append({"compound": case["name"], "model": model_name, "condition": CONDITION, "set_id": set_id, "heads": json.dumps(random_set_map[set_id]), "kl": result["kl_base_to_ablated"], "top_changed": result["baseline_top"][0][0] != result["ablated_top"][0][0], "bigram_count": count, "log_frequency": np.log1p(count)})
    pd.DataFrame(random_rows).to_csv(random_output_path, index=False)
random_results = pd.DataFrame(random_rows)
if len(random_results) != len(test_cases) * N_RANDOM_SETS:
    raise ValueError(f"Expected {len(test_cases) * N_RANDOM_SETS} random-control rows; found {len(random_results)}.")
print(f"Saved all random controls: {random_output_path}")

In [ ]:
# Cell 9: Run the registered directional test and summarize controls
selected_kl = primary_results["selected_kl"].to_numpy(dtype=float)
log_frequency = primary_results["log_frequency"].to_numpy(dtype=float)
observed_rho = float(spearmanr(log_frequency, selected_kl).statistic)
if np.isfinite(observed_rho):
    rng_permutation = np.random.default_rng(RANDOM_SEED)
    permuted_rhos = np.array([spearmanr(rng_permutation.permutation(log_frequency), selected_kl).statistic for _ in range(N_PERMUTATIONS)], dtype=float)
    permutation_p = (1 + int(np.sum(permuted_rhos <= observed_rho))) / (N_PERMUTATIONS + 1)
else:
    permutation_p = 1.0
random_summary_rows = []
for set_id, group in random_results.groupby("set_id", sort=True):
    rho = float(spearmanr(group["log_frequency"], group["kl"]).statistic)
    random_summary_rows.append({"set_id": set_id, "heads": group["heads"].iloc[0], "rho_frequency_kl": rho, "mean_kl": float(group["kl"].mean()), "median_kl": float(group["kl"].median()), "top_changes": int(group["top_changed"].sum())})
random_set_summary = pd.DataFrame(random_summary_rows)
random_set_summary_path = output_dir / f"{model_name}-{CONDITION}-registered-random-set-summary.csv"
random_set_summary.to_csv(random_set_summary_path, index=False)
finite_random_rhos = random_set_summary["rho_frequency_kl"].dropna().to_numpy()
random_tail_p = ((1 + int(np.sum(finite_random_rhos <= observed_rho))) / (len(finite_random_rhos) + 1)) if np.isfinite(observed_rho) else 1.0
summary = pd.DataFrame([{"model": model_name, "condition": CONDITION, "random_seed": RANDOM_SEED, "n_selection": 25, "n_test": len(primary_results), "selected_rho_frequency_kl": observed_rho, "selected_permutation_p_one_sided": permutation_p, "n_permutations": N_PERMUTATIONS, "selected_mean_kl": float(primary_results["selected_kl"].mean()), "selected_median_kl": float(primary_results["selected_kl"].median()), "selected_top_changes": int(primary_results["selected_top_changed"].sum()), "empty_max_abs_kl": empty_max, "positive_control_max_kl": positive_max, "positive_control_median_kl": float(primary_results["positive_control_kl"].median()), "positive_control_top_changes": int(primary_results["positive_control_top_changed"].sum()), "n_random_sets": N_RANDOM_SETS, "n_finite_random_rhos": len(finite_random_rhos), "random_rho_median": float(np.median(finite_random_rhos)) if len(finite_random_rhos) else np.nan, "selected_rho_random_tail_p": random_tail_p}])
summary_path = output_dir / f"{model_name}-{CONDITION}-registered-summary.csv"
summary.to_csv(summary_path, index=False)
print("Registered selected-head result:")
print(f"  rho(frequency, KL) = {observed_rho:+.3f}")
print(f"  one-sided permutation p = {permutation_p:.4g}")
print(f"  random-set tail p = {random_tail_p:.4g} ({len(finite_random_rhos)} finite random correlations)")
print(f"  selected-set top-token changes = {int(primary_results['selected_top_changed'].sum())}/{len(primary_results)}")
print("\nInterpretation branches:")
print("- Negative association: evidence consistent with compensatory, load-bearing late processing.")
print("- Flat association: frequency-sensitive activity is not shown to be causally necessary under this intervention.")
print("- This notebook measures distributional change, not task accuracy.")
display(summary)

In [ ]:
# Cell 10: Write the contemporaneous manifest
manifest_path = write_registered_ablation_manifest(
    project_root=PROJECT_ROOT, primary_csv=primary_output_path, random_controls_csv=random_output_path,
    random_set_summary_csv=random_set_summary_path, head_sets_csv=head_sets_path, summary_csv=summary_path,
    split_path=split_path, candidate_path=candidate_path, amendment_path=amendment_path,
    notebook_path=PROJECT_ROOT / "notebooks" / "frequency-head-ablation-pythia.ipynb",
    intervention_code_path=PROJECT_ROOT / "src" / "qk_ov.py",
    model_metadata_manifest=PROJECT_ROOT / "results" / "effective_binding" / "pythia" / model_name / CONDITION / f"{model_name}-{CONDITION}-effective-binding.md",
    model=model, selected_heads=selected_heads, positive_control_heads=positive_control_heads,
    random_seed=RANDOM_SEED, n_permutations=N_PERMUTATIONS,
)
print(f"Manifest: {manifest_path}")
print("No Git commands were run.")

### Delete the model and clear memory

In [ ]:
# Cell 11: Cleanup
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()
print("Memory cleared")

## Final step: save the registered results to GitHub

Run this cell only after the manifest cell succeeds. It stages only the six new registered-result artifacts, commits them, and pushes them so they survive the Colab runtime.


In [ ]:
# Cell 12: Save completed registered results to GitHub
import subprocess

if not IN_COLAB:
    raise RuntimeError("This save-and-push cell is intended for the Colab run only.")

result_paths = [
    head_sets_path,
    primary_output_path,
    random_output_path,
    random_set_summary_path,
    summary_path,
    manifest_path,
]
missing_results = [path for path in result_paths if not path.exists()]
if missing_results:
    raise FileNotFoundError(f"Cannot save an incomplete run. Missing: {missing_results}")

relative_paths = [str(path.relative_to(PROJECT_ROOT)) for path in result_paths]
subprocess.run(["git", "config", "user.email", "trisha@trishasalas.com"], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "config", "user.name", "Trisha Salas"], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "add", "--", *relative_paths], cwd=PROJECT_ROOT, check=True)
staged = subprocess.run(["git", "diff", "--cached", "--quiet", "--", *relative_paths], cwd=PROJECT_ROOT)
if staged.returncode == 0:
    subprocess.run(["git", "push", "origin", "main"], cwd=PROJECT_ROOT, check=True)
    print("Registered result files were already committed; GitHub is up to date.")
elif staged.returncode == 1:
    subprocess.run(["git", "commit", "-m", "results: registered Pythia-2.8B frequency-head ablation", "--", *relative_paths], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "push", "origin", "main"], cwd=PROJECT_ROOT, check=True)
    print("Registered results committed and pushed to GitHub.")
else:
    raise RuntimeError("Git could not inspect the staged result files.")